# Build Constructors Dimension
1. Read silver `constructors` table
2. Read gold `ref_nationality_region` table
3. Join the data from `constructors` with `ref_nationality_region` using `nationality`
4. Select the required columns
    - constructors.constructor_id
    - constructors.constructor_name
    - constructors.nationality
    - ref_nationality_region.region
5. Write the transformed data to gold `dim_constructors` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read source tables
- `silver.constructors`
- `gold.ref_nationality_region`

In [0]:
constructors_df = spark.table(f"{catalog_name}.{silver_schema}.constructors")
ref_nat_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

#### Step 2 - Join `constructors` with `nationality_region_df` using `nationality`
Select the following columns   
1. constructors.constructor_id 
2. constructors.constructor_name 
3. constructors.nationality 
4. ref_nationality_region.region

In [0]:
dim_constructors_df = (
        constructors_df
            .join(
                ref_nat_region_df,
                constructors_df.nationality == ref_nat_region_df.nationality,
                "left"
            )
            .select(
                constructors_df.constructor_id,
                constructors_df.constructor_name,
                constructors_df.nationality,
                ref_nat_region_df.region.alias("nationality_region")

            )
)

display(dim_constructors_df)

#### Step 3 - Write the transformed data to the `gold` `dim_constructors` table

In [0]:
(
    dim_constructors_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

display(spark.table(target_table))